In [1]:
import numpy as np
from phoenix import Hamiltonian
from phoenix.primitives.simplification import heuristic_bsf_cost, CLIFFORD_OPTIONS
from qiskit.quantum_info import Clifford
from qiskit.circuit.library import CXGate
from qiskit.circuit import QuantumCircuit
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter, SuzukiTrotter, ProductFormula, EvolutionSynthesis


In [2]:


# # 1. Define Hamiltonian as SparsePauliOp
# H = SparsePauliOp.from_list([("-XY", np.pi), ("ZZ", 0.5)])

# # 2. Create  (exp(-iHt))
# # This gate represents the unitary evolution U(t) = e^{-iHt}
# evo_gate = PauliEvolutionGate(H, time=1.0)

# # 3. Create QuantumCircuit and append the gate
# qc = QuantumCircuit(2)
# qc.append(evo_gate, [0, 1])

# # 4. Synthesize (decompose) into basis gates
# # By default, Qiskit might use LieTrotter or MatrixExponential depending on context/settings
# # We can explicitly decompose it using a synthesis plugin or .decompose()
# print("Original Circuit (High-level):")
# print(qc.draw())

# print("\nDecomposed Circuit (LieTrotter):")
# # Using LieTrotter synthesis
# trotter_factory = LieTrotter()
# qc_synthesized = trotter_factory.synthesize(evo_gate)
# qc_synthesized.draw(fold=-1)

In [3]:
ham = Hamiltonian(['XXXZIYZI', 'YXXZIYYI', 'ZXXZIYZI'], [-0.0125, -0.0125, -0.0125])
ham.print_tableau()

+----------+-----------------+-----------------+---+
|  Pauli   |      X part     |      Z part     | s |
+----------+-----------------+-----------------+---+
| IZYIZXXX | 0 0 1 0 0 1 1 1 | 0 1 1 0 1 0 0 0 | 0 |
| IYYIZXXY | 0 1 1 0 0 1 1 1 | 0 1 1 0 1 0 0 1 | 0 |
| IZYIZXXZ | 0 0 1 0 0 1 1 0 | 0 1 1 0 1 0 0 1 | 0 |
+----------+-----------------+-----------------+---+


In [4]:
Hamiltonian(['XZ']).print_tableau()

+-------+--------+--------+---+
| Pauli | X part | Z part | s |
+-------+--------+--------+---+
|   ZX  |  0 1   |  1 0   | 0 |
+-------+--------+--------+---+


In [5]:
Hamiltonian(['XZ']).to_matrix()

array([[ 0.+0.j,  0.+0.j,  1.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j],
       [ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j, -1.+0.j,  0.+0.j,  0.+0.j]])

In [7]:
heuristic_bsf_cost(ham.paulis.x, ham.paulis.z)

85.0

In [8]:
ham.group_same_weights()

[SparsePauliOp(['XXXZIYZI', 'YXXZIYYI', 'ZXXZIYZI'],
               coeffs=[-0.0125+0.j, -0.0125+0.j, -0.0125+0.j])]

In [19]:
from phoenix.primitives import simplification


ham0 = ham.group_same_weights()[0]

ham_, simp_steps = simplification.simplify_hamiltonian(ham0)

In [20]:
ham_.print_tableau()
print(ham_.paulis.to_labels())

+----------+-----------------+-----------------+---+
|  Pauli   |      X part     |      Z part     | s |
+----------+-----------------+-----------------+---+
| IZIIIIIX | 0 0 0 0 0 0 0 1 | 0 1 0 0 0 0 0 0 | 0 |
| IYIIIIIY | 0 1 0 0 0 0 0 1 | 0 1 0 0 0 0 0 1 | 0 |
| IZIIIIIZ | 0 0 0 0 0 0 0 0 | 0 1 0 0 0 0 0 1 | 0 |
+----------+-----------------+-----------------+---+
['XIIIIIZI', 'YIIIIIYI', 'ZIIIIIZI']


In [21]:
qc_origin = QuantumCircuit(ham.num_qubits)
qc_origin.append(PauliEvolutionGate(ham), range(ham.num_qubits))
qc_origin.decompose().draw(fold=-1)

q_0: ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                               ┌───┐┌────────────┐┌───┐┌────┐                                                  ┌───┐┌────────────┐┌───┐┌──────┐                                                    ┌───┐┌────────────┐┌───┐                            
q_1: ──────────────────────────┤ X ├┤ Rz(-0.025) ├┤ X ├┤ √X ├──────────────────────────────────────────────────┤ X ├┤ Rz(-0.025) ├┤ X ├┤ √Xdg ├────────────────────────────────────────────────────┤ X ├┤ Rz(-0.025) ├┤ X ├────────────────────────────
     ┌────┐               ┌───┐└─┬─┘└────────────┘└─┬─┘├───┬┘┌──────┐┌────┐                               ┌───┐└─┬─┘└────────────┘└─┬─┘└┬───┬─┘┌──────┐┌────┐                                 ┌───┐└─┬─┘└────────────┘└─┬─┘┌───┐┌──────┐               
q_2: ┤ √X ├───────────────┤ X ├──■──────────────────■──┤ X ├─┤ √Xdg ├┤ √X ├───────────────────────────────┤ X ├──■──────────────────■───┤ X ├──┤ √Xdg ├┤ √X ├─────────────────────────────────┤ X ├──■──────────────────■──┤ X ├┤ √Xdg ├───────────────
     └────┘               └─┬─┘                        └─┬─┘ └──────┘└────┘                               └─┬─┘                         └─┬─┘  └──────┘└────┘                                 └─┬─┘                        └─┬─┘└──────┘               
q_3: ───────────────────────┼────────────────────────────┼──────────────────────────────────────────────────┼─────────────────────────────┼─────────────────────────────────────────────────────┼────────────────────────────┼─────────────────────────
                     ┌───┐  │                            │    ┌───┐                                  ┌───┐  │                             │     ┌───┐                                    ┌───┐  │                            │   ┌───┐                 
q_4: ────────────────┤ X ├──■────────────────────────────■────┤ X ├──────────────────────────────────┤ X ├──■─────────────────────────────■─────┤ X ├────────────────────────────────────┤ X ├──■────────────────────────────■───┤ X ├─────────────────
     ┌───┐      ┌───┐└─┬─┘                                    └─┬─┘  ┌───┐ ┌───┐┌───┐           ┌───┐└─┬─┘                                      └─┬─┘  ┌───┐ ┌───┐ ┌───┐            ┌───┐└─┬─┘                                   └─┬─┘  ┌───┐┌───┐     
q_5: ┤ H ├──────┤ X ├──■────────────────────────────────────────■────┤ X ├─┤ H ├┤ H ├───────────┤ X ├──■──────────────────────────────────────────■────┤ X ├─┤ H ├─┤ H ├────────────┤ X ├──■───────────────────────────────────────■────┤ X ├┤ H ├─────
     ├───┤ ┌───┐└─┬─┘                                                └─┬─┘ ├───┤├───┤┌───┐ ┌───┐└─┬─┘                                                  └─┬─┘ ├───┤ ├───┤  ┌───┐┌───┐└─┬─┘                                               └─┬─┘├───┤┌───┐
q_6: ┤ H ├─┤ X ├──■────────────────────────────────────────────────────■───┤ X ├┤ H ├┤ H ├─┤ X ├──■──────────────────────────────────────────────────────■───┤ X ├─┤ H ├──┤ H ├┤ X ├──■───────────────────────────────────────────────────■──┤ X ├┤ H ├
     ├───┤ └─┬─┘                                                           └─┬─┘├───┤├───┴┐└─┬─┘                                                             └─┬─┘┌┴───┴─┐└───┘└─┬─┘                                                         └─┬─┘└───┘
q_7: ┤ H ├───■───────────────────────────────────────────────────────────────■──┤ H ├┤ √X ├──■─────────────────────────────────────────────────────────────────■──┤ √Xdg ├───────■─────────────────────────────────────────────────────────────■───────
     └───┘                                                                      └───┘└────┘                                                                       └──────┘

In [49]:
qc = QuantumCircuit(2)
qc.append(CXGate(), (0,1))
qc.append(PauliEvolutionGate(Hamiltonian(['XI'])), (0,1))
qc.decompose('PauliEvolution').draw()

q_0: ──■───────────
     ┌─┴─┐┌───────┐
q_1: ┤ X ├┤ Rx(2) ├
     └───┘└───────┘

In [43]:
PauliEvolutionGate(Hamiltonian(['IX']))

Instruction(name='PauliEvolution', num_qubits=2, num_clbits=0, params=[1.0])

In [54]:
from phoenix.basics import CNOTEquivCliffordGate


g1 = CNOTEquivCliffordGate('X', 'Z')
g2 = g1.copy()

In [ ]:
g1.reverse_ops

(13566928768, 13566933568)

In [31]:
for step in simp_steps:
    print(step.local_hamiltonian.total_weight)

0
0
0
0


In [ ]:
qc 

Instruction(name='PauliEvolution', num_qubits=2, num_clbits=0, params=[1.0])

In [ ]:
qc_simp = QuantumCircuit(ham_.num_qubits)

qc_simp.append(PauliEvolutionGate(ham_), range(ham_.num_qubits))


for step in simp_steps:
    ...







qc_simp.decompose().draw(fold=-1)

q_0: ─────────────────────────────────────────────
     ┌──────────────┐┌──────────────┐             
q_1: ┤0             ├┤0             ├─■───────────
     │              ││              │ │           
q_2: ┤              ├┤              ├─┼───────────
     │              ││              │ │           
q_3: ┤              ├┤              ├─┼───────────
     │              ││              │ │           
q_4: ┤  Rzx(-0.025) ├┤  Ryy(-0.025) ├─┼───────────
     │              ││              │ │           
q_5: ┤              ├┤              ├─┼───────────
     │              ││              │ │           
q_6: ┤              ├┤              ├─┼───────────
     │              ││              │ │ZZ(-0.025) 
q_7: ┤1             ├┤1             ├─■───────────
     └──────────────┘└──────────────┘

In [13]:
ham_.paulis.to_labels()

['XIIIIIZI', 'YIIIIIYI', 'ZIIIIIZI']

In [14]:
ham_.active_qubits

array([1, 7])

In [15]:
ham_.print_tableau()

+----------+-----------------+-----------------+---+
|  Pauli   |      X part     |      Z part     | s |
+----------+-----------------+-----------------+---+
| IZIIIIIX | 0 0 0 0 0 0 0 1 | 0 1 0 0 0 0 0 0 | 0 |
| IYIIIIIY | 0 1 0 0 0 0 0 1 | 0 1 0 0 0 0 0 1 | 0 |
| IZIIIIIZ | 0 0 0 0 0 0 0 0 | 0 1 0 0 0 0 0 1 | 0 |
+----------+-----------------+-----------------+---+


In [34]:
ham_.with_ops

array([[False,  True, False, False, False, False, False,  True],
       [False,  True, False, False, False, False, False,  True],
       [False,  True, False, False, False, False, False,  True]])

In [ ]:

# # 1. Define Hamiltonian as SparsePauliOp
# H = SparsePauliOp.from_list([("-XY", np.pi), ("ZZ", 0.5)])

# # 2. Create  (exp(-iHt))
# # This gate represents the unitary evolution U(t) = e^{-iHt}
# evo_gate = PauliEvolutionGate(H, time=1.0)

# # 3. Create QuantumCircuit and append the gate
# qc = QuantumCircuit(2)
# qc.append(evo_gate, [0, 1])

# # 4. Synthesize (decompose) into basis gates
# # By default, Qiskit might use LieTrotter or MatrixExponential depending on context/settings
# # We can explicitly decompose it using a synthesis plugin or .decompose()
# print("Original Circuit (High-level):")
# print(qc.draw())

# print("\nDecomposed Circuit (LieTrotter):")
# # Using LieTrotter synthesis
# trotter_factory = LieTrotter()
# qc_synthesized = trotter_factory.synthesize(evo_gate)
# qc_synthesized.draw(fold=-1)


In [7]:
len(simp_steps)

4

In [18]:
from itertools import combinations   
qubit_pairs = sorted(combinations(ham.active_qubits, 2), key=lambda idx: (idx[0] % 2))
qubit_pairs = [pair for pair in qubit_pairs if pair != (-1, -1)]

In [25]:
len(qubit_pairs)




15

In [28]:
for i in [1,2,3]:
    for s in ['a', 'b']:
        print(i, s)

1 a
1 b
2 a
2 b
3 a
3 b


In [29]:
[(i, s) for i in [1,2,3] for s in ['a', 'b']]

[(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b'), (3, 'a'), (3, 'b')]

In [24]:
qubit_pairs

[(2, 4),
 (2, 5),
 (2, 6),
 (2, 7),
 (4, 5),
 (4, 6),
 (4, 7),
 (6, 7),
 (1, 2),
 (1, 4),
 (1, 5),
 (1, 6),
 (1, 7),
 (5, 6),
 (5, 7)]

In [11]:
simp_steps

[SimplificationStep(clifford=Instruction(name='cyy', num_qubits=2, num_clbits=0, params=[]), local_hamiltonian=SparsePauliOp(['IIIIIIII'],
               coeffs=[0.+0.j]), qubits=(2, 4)),
 SimplificationStep(clifford=Instruction(name='cxx', num_qubits=2, num_clbits=0, params=[]), local_hamiltonian=SparsePauliOp(['IIIIIIII'],
               coeffs=[0.+0.j]), qubits=(4, 5)),
 SimplificationStep(clifford=Instruction(name='cxx', num_qubits=2, num_clbits=0, params=[]), local_hamiltonian=SparsePauliOp(['IIIIIIII'],
               coeffs=[0.+0.j]), qubits=(4, 6)),
 SimplificationStep(clifford=Instruction(name='cxz', num_qubits=2, num_clbits=0, params=[]), local_hamiltonian=SparsePauliOp(['IIIIIIII'],
               coeffs=[0.+0.j]), qubits=(1, 4))]

In [8]:
ham_.paulis.to_labels()

['XIIIIIZI', 'YIIIIIYI', 'ZIIIIIZI']

In [10]:
from regulus.phoenix.simplification import simplify_bsf
from regulus.phoenix.simplification import heuristic_bsf_cost as hbc
from regulus.models import BSF


In [12]:
bsf = BSF(ham.paulis.to_labels(), ham.coeffs)
b, cliff_with_locals = simplify_bsf(bsf)

/Users/anan/git-projects/quantum/reqisc_asplos_2026/regulus/models/paulis.py:27: ComplexWarning: Casting complex values to real discards the imaginary part
  self.signs = np.atleast_1d(signs) if signs is not None else np.zeros_like(self.coeffs).astype(int)


In [13]:
b.paulis

['XIIIIIZI', 'YIIIIIYI', 'ZIIIIIZI']

In [14]:
cliff_with_locals

[(C(X, Y) @ (3, 5), BSF(size=[0, 8], num_nonlocals=0, total_weight=0)),
 (C(X, X) @ (1, 3), BSF(size=[0, 8], num_nonlocals=0, total_weight=0)),
 (C(X, X) @ (2, 3), BSF(size=[0, 8], num_nonlocals=0, total_weight=0)),
 (C(Z, X) @ (3, 6), BSF(size=[0, 8], num_nonlocals=0, total_weight=0))]

In [15]:
hbc(bsf)

array(85.)